# systemgmmkit quickstart

This notebook is a package-scoped tour of `systemgmmkit`: panel-data setup, robust OLS, fixed/random effects, post-estimation, and panel-aware forecast validation. It uses deterministic simulated data so it can run on Kaggle or Google Colab without external datasets.

What this notebook is: a reproducible cloud demo for users and reviewers.  
What it is not: a full dynamic-GMM parity certificate or paper artifact.

## Install

Kaggle and Colab runtimes are usually clean. If you are running from a local checkout, skip this cell and make sure the source tree is on `PYTHONPATH`.

In [ ]:
%pip install -q systemgmmkit

## Shared imports and deterministic panel data

In [ ]:
import numpy as np
import pandas as pd

import systemgmmkit as sgk
from systemgmmkit.ml import (
    PanelTimeSeriesSplit,
    compare_models,
    cross_validate_panel,
    panel_train_test_split,
)

SEED = 20260730
rng = np.random.default_rng(SEED)

rows = []
for firm in range(1, 31):
    firm_effect = rng.normal(scale=0.35)
    previous_growth = rng.normal(scale=0.2)
    for year in range(2012, 2022):
        investment = rng.normal(loc=1.0 + 0.04 * (year - 2012), scale=0.35)
        leverage = rng.uniform(0.2, 0.8)
        size = rng.normal(loc=firm / 10.0, scale=0.25)
        shock = rng.normal(scale=0.18)
        growth = 0.35 * previous_growth + 0.65 * investment - 0.30 * leverage + 0.12 * size + firm_effect + shock
        rows.append({
            "firm_id": firm,
            "year": year,
            "growth": growth,
            "L1_growth": previous_growth,
            "investment": investment,
            "leverage": leverage,
            "size": size,
        })
        previous_growth = growth

panel = pd.DataFrame(rows)
print(panel.shape)
display(panel.head())

## 1. Robust pooled panel OLS

This is the simplest baseline. It gives users a known starting point before moving to fixed/random effects or GMM specifications.

In [ ]:
pooled_spec = sgk.OLSSpec(
    dependent="growth",
    regressors=["L1_growth", "investment", "leverage", "size"],
    covariance="robust",
    name="pooled_dynamic_panel_ols",
)
pooled = sgk.run_ols(pooled_spec, panel, entity="firm_id", time="year")

post = sgk.quick_postestimation(
    pooled,
    panel,
    y="growth",
    lincoms={"investment_minus_leverage": "investment - leverage"},
    wald_tests={"joint_investment_leverage": "investment = 0, leverage = 0"},
)

display(pooled.params.round(4).to_frame("estimate"))
display(pd.Series(post.metrics).round(6).to_frame("value"))
display(post.linear_combinations.round(4))
display(post.wald_tests.round(4))

assert np.isfinite(post.metrics["rmse"])
assert post.linear_combinations is not None
assert post.wald_tests is not None

## 2. Fixed and random effects specifications

The same panel can be estimated with entity-aware specifications. This is the natural bridge from ordinary panel regression to dynamic-panel workflows.

In [ ]:
fe_spec = sgk.FixedEffectsSpec(
    dependent="growth",
    regressors=["investment", "leverage", "size"],
    entity_effects=True,
    covariance="robust",
    name="entity_fixed_effects",
)
re_spec = sgk.RandomEffectsSpec(
    dependent="growth",
    regressors=["investment", "leverage", "size"],
    covariance="robust",
    name="random_effects",
)

fe = sgk.run_fixed_effects(fe_spec, panel, entity="firm_id", time="year")
re = sgk.run_random_effects(re_spec, panel, entity="firm_id", time="year")

comparison = pd.concat(
    {
        "pooled_ols": pooled.params,
        "fixed_effects": fe.params,
        "random_effects": re.params,
    },
    axis=1,
)
display(comparison.round(4))

assert fe.nobs > 0 and re.nobs > 0
assert set(["investment", "leverage", "size"]).issubset(fe.params.index)

## 3. Panel-aware train/test split and cross-validation

The ML layer is useful when the question is predictive performance rather than structural interpretation. The split respects panel time ordering.

In [ ]:
def fit_forecast_model(data: pd.DataFrame):
    return sgk.run_ols(
        sgk.OLSSpec(
            dependent="growth",
            regressors=["L1_growth", "investment", "leverage", "size"],
            covariance="robust",
            name="forecast_ols",
        ),
        data,
        entity="firm_id",
        time="year",
    )

train, test = panel_train_test_split(panel, time="year", test_size=2)
train_result = fit_forecast_model(train)

holdout = compare_models(
    {"dynamic pooled OLS": train_result},
    test,
    y="growth",
    predict_kwargs={"strict": False},
)
cv = cross_validate_panel(
    estimator=fit_forecast_model,
    data=panel,
    y="growth",
    time="year",
    cv=PanelTimeSeriesSplit(n_splits=3, min_train_periods=5, test_periods=1),
    predict_kwargs={"strict": False},
)

display(holdout.round(4))
display(cv.round(4))

assert len(holdout) == 1
assert len(cv) == 3

## 4. Interpretation checklist

For a public notebook, keep the claims clear:

- pooled, fixed-effects, and random-effects estimates answer different questions;
- panel-aware validation is about forecasting discipline, not causal identification;
- dynamic GMM validation requires AR tests, Hansen/Sargan evidence, instrument count checks, and parity artifacts, which belong in the package validation and paper workflow.